In [1]:
"""
Two-Drone Coverage Simulation: Reinforcement Learning + Evolutionary Strategy
==============================================================================

Setup
-----
- A square area is split into a 4x4 grid of 16 cells, labeled 1..16.
- Two drones fly at fixed height, one over cell c1 and one over cell c2.
- 3 cell-phone users sit at fixed (random) cell locations for the duration
  of an episode.
- Each period t the RL agent issues one move command per drone:
  N (north), S (south), E (east), W (west), or STAY.
  A move that would leave the 4x4 area is ignored (the drone stays put).
- The "state" the agent sees each period is (c1, c2, sigma, z):
    c1, c2 : the two drones' current cells (1..16)
    sigma  : the current coverage/step-size parameter of the evolutionary
             strategy (controls how forgiving distance is for connectivity)
    z      : the most recent ES mutation sample (used to adapt sigma)
- Reward each period is the *improvement* in average communication
  probability across the 3 users, so the RL agent is pushed to keep moving
  the drones toward configurations that serve users better.

Modeling choices (since the prompt leaves the exact physics open)
-------------------------------------------------------------------
1. Communication probability between a drone and a user is modeled as a
   Gaussian function of grid distance:
       P(dist) = exp(-dist^2 / (2*sigma^2))
   A user connects through whichever of the two drones gives it the higher
   probability (i.e. the nearer one). The per-period reward signal for the
   whole configuration is the mean of the 3 users' connection
   probabilities.
2. sigma is treated as a difficulty/tolerance dial adapted by a (1+1)
   self-adaptive Evolutionary Strategy: every `window_size` periods we look
   at the recent success rate (fraction of periods where coverage
   improved) and apply a 1/5-success rule, drawing a mutation
   z ~ N(0, tau). When the agent is succeeding (success rate > 1/5) sigma
   is *tightened* (exp(-|z|)) -- the tolerance gets stricter because the
   agent is coping well, pushing it toward genuinely precise positioning
   instead of coasting on a generous tolerance. When the agent is
   struggling, sigma is *relaxed* (exp(+|z|)) so the problem gets easier
   again. Consequence: once the agent holds a good/optimal configuration
   and coverage plateaus, the success rate drops, so sigma stops being
   squeezed and instead relaxes -- coverage is not driven down by an
   ever-shrinking sigma the way an earlier version of this script did.
3. Reward given to the RL agent each period is the *absolute* mean
   coverage probability P_comm (not its period-to-period change). Holding
   a good configuration keeps paying full reward, so the Q-learner has no
   incentive to wander away from a plateaued optimum -- which was the
   other cause of the earlier "coverage decays over the episode" symptom
   (a delta-based reward gives ~0 reward once P_comm stops improving,
   even while positions are still great).
4. The RL controller itself is tabular Q-learning over the discretized
   state (c1, c2, sigma-bin, z-bin) and the 25 joint drone actions
   (5 actions per drone).

This is intentionally a compact, well-commented reference implementation --
swap in a deep RL agent, a different propagation model, or a richer ES
update rule as needed.
"""

import random
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --------------------------------------------------------------------------
# Grid geometry
# --------------------------------------------------------------------------
GRID_SIZE = 4
N_CELLS = GRID_SIZE * GRID_SIZE
ACTIONS = ["N", "S", "E", "W", "STAY"]


def cell_to_rc(cell):
    idx = cell - 1
    return idx // GRID_SIZE, idx % GRID_SIZE


def rc_to_cell(r, c):
    return r * GRID_SIZE + c + 1


def move(cell, action):
    r, c = cell_to_rc(cell)
    if action == "N":
        r = max(0, r - 1)
    elif action == "S":
        r = min(GRID_SIZE - 1, r + 1)
    elif action == "E":
        c = min(GRID_SIZE - 1, c + 1)
    elif action == "W":
        c = max(0, c - 1)
    # STAY -> unchanged; out-of-bounds moves are clipped (drone stays put)
    return rc_to_cell(r, c)


def distance(cell_a, cell_b):
    ra, ca = cell_to_rc(cell_a)
    rb, cb = cell_to_rc(cell_b)
    return float(np.hypot(ra - rb, ca - cb))


# --------------------------------------------------------------------------
# Communication model
# --------------------------------------------------------------------------
def comm_prob(dist, sigma):
    return float(np.exp(-(dist ** 2) / (2.0 * sigma ** 2)))


def coverage_probability(c1, c2, users, sigma):
    per_user = []
    for u in users:
        p = max(comm_prob(distance(c1, u), sigma),
                comm_prob(distance(c2, u), sigma))
        per_user.append(p)
    return float(np.mean(per_user)), per_user


# --------------------------------------------------------------------------
# Discretization helpers for the state fed to the tabular Q-learner
# --------------------------------------------------------------------------
SIGMA_MIN, SIGMA_MAX = 0.5, 4.0
_SIGMA_EDGES = np.linspace(SIGMA_MIN, SIGMA_MAX, 5)   # -> 4 bins
_Z_EDGES = np.array([-1.0, -0.2, 0.2, 1.0])           # -> 4 bins


def sigma_bin(sigma):
    return int(np.clip(np.digitize(sigma, _SIGMA_EDGES) - 1, 0, 3))


def z_bin(z):
    return int(np.clip(np.digitize(z, _Z_EDGES), 0, 3))


# --------------------------------------------------------------------------
# Environment: holds drone positions, users, and the ES state (sigma, z)
# --------------------------------------------------------------------------
class DroneEnv:
    def __init__(self, tau=0.3, window_size=10):
        self.tau = tau
        self.window_size = window_size

    def reset(self, users, c1=None, c2=None, sigma=1.5):
        self.users = list(users)
        self.c1 = c1 if c1 is not None else random.randint(1, N_CELLS)
        self.c2 = c2 if c2 is not None else random.randint(1, N_CELLS)
        self.sigma = sigma
        self.z = 0.0
        self.success_window = []
        self.t = 0
        self.prev_p, _ = coverage_probability(self.c1, self.c2, self.users, self.sigma)
        return self.get_state()

    def get_state(self):
        return (self.c1, self.c2, sigma_bin(self.sigma), z_bin(self.z))

    def step(self, a1, a2):
        self.c1 = move(self.c1, a1)
        self.c2 = move(self.c2, a2)
        p, per_user = coverage_probability(self.c1, self.c2, self.users, self.sigma)

        # RL reward = absolute coverage probability (not the delta), so
        # holding a good configuration keeps paying full reward instead of
        # decaying to ~0 once improvement stops.
        reward = p

        # The ES's own "success" signal is still improvement-based -- that
        # is what the 1/5 rule is meant to track.
        self.success_window.append(p > self.prev_p)
        if len(self.success_window) > self.window_size:
            self.success_window.pop(0)

        # Self-adaptive (1/5-success-rule) update of the ES parameter sigma.
        # Doing well -> tighten the tolerance (harder). Struggling -> relax
        # it (easier). This means a plateau at a good configuration stops
        # squeezing sigma further (success rate falls, so sigma relaxes
        # instead of keeps shrinking), which is what previously drove
        # coverage probability down over the course of an episode.
        if len(self.success_window) == self.window_size:
            success_rate = float(np.mean(self.success_window))
            self.z = float(np.random.normal(0.0, self.tau))
            if success_rate > 0.2:
                self.sigma *= np.exp(-abs(self.z))  # tighten: demand precision
            else:
                self.sigma *= np.exp(abs(self.z))   # relax: make it easier
            self.sigma = float(np.clip(self.sigma, SIGMA_MIN, SIGMA_MAX))

        self.prev_p = p
        self.t += 1
        return self.get_state(), reward, p, per_user


# --------------------------------------------------------------------------
# Tabular Q-learning agent over the 25 joint (drone1, drone2) actions
# --------------------------------------------------------------------------
class QLearningAgent:
    def __init__(self, alpha=0.2, gamma=0.9, epsilon=0.3,
                 epsilon_decay=0.995, epsilon_min=0.05):
        self.q = {}
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.action_pairs = [(a1, a2) for a1 in ACTIONS for a2 in ACTIONS]

    def _row(self, state):
        if state not in self.q:
            self.q[state] = np.zeros(len(self.action_pairs))
        return self.q[state]

    def choose_action(self, state):
        if random.random() < self.epsilon:
            idx = random.randrange(len(self.action_pairs))
        else:
            idx = int(np.argmax(self._row(state)))
        return idx, self.action_pairs[idx]

    def update(self, state, action_idx, reward, next_state):
        row = self._row(state)
        best_next = np.max(self._row(next_state))
        row[action_idx] += self.alpha * (reward + self.gamma * best_next - row[action_idx])

    def decay(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


# --------------------------------------------------------------------------
# Training loop: random user placements each episode
# --------------------------------------------------------------------------
def train(n_episodes=800, periods_per_episode=15, seed=0):
    random.seed(seed)
    np.random.seed(seed)
    env = DroneEnv()
    agent = QLearningAgent()
    final_period_probs = []

    for ep in range(n_episodes):
        users = random.sample(range(1, N_CELLS + 1), 3)
        state = env.reset(users=users, sigma=1.5)
        last_p = None
        for _ in range(periods_per_episode):
            idx, (a1, a2) = agent.choose_action(state)
            next_state, reward, p, _ = env.step(a1, a2)
            agent.update(state, idx, reward, next_state)
            state = next_state
            last_p = p
        agent.decay()
        final_period_probs.append(last_p)

    return env, agent, final_period_probs


# --------------------------------------------------------------------------
# Greedy evaluation on a chosen user configuration, keeping full trajectory
# --------------------------------------------------------------------------
def evaluate(agent, users, periods=15, c1_start=1, c2_start=16, sigma0=1.5, seed=123):
    random.seed(seed)
    env = DroneEnv()
    state = env.reset(users=users, c1=c1_start, c2=c2_start, sigma=sigma0)

    path1, path2 = [env.c1], [env.c2]
    probs, sigmas = [env.prev_p], [env.sigma]

    saved_eps = agent.epsilon
    agent.epsilon = 0.0  # act greedily for evaluation
    for _ in range(periods):
        idx, (a1, a2) = agent.choose_action(state)
        state, _, p, _ = env.step(a1, a2)
        path1.append(env.c1)
        path2.append(env.c2)
        probs.append(p)
        sigmas.append(env.sigma)
    agent.epsilon = saved_eps

    return path1, path2, probs, sigmas


# --------------------------------------------------------------------------
# Plotting
# --------------------------------------------------------------------------
def plot_simulation(path1, path2, probs, sigmas, users, training_curve, outfile):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # --- Panel 1: grid + drone trajectories + user positions -------------
    ax = axes[0]
    for i in range(GRID_SIZE + 1):
        ax.axhline(i, color="lightgray", lw=1)
        ax.axvline(i, color="lightgray", lw=1)
    for cell in range(1, N_CELLS + 1):
        r, c = cell_to_rc(cell)
        ax.text(c + 0.5, GRID_SIZE - r - 0.5, str(cell),
                ha="center", va="center", color="lightgray", fontsize=9)

    def cell_xy(cell):
        r, c = cell_to_rc(cell)
        return c + 0.5, GRID_SIZE - r - 0.5

    xs1, ys1 = zip(*[cell_xy(c) for c in path1])
    xs2, ys2 = zip(*[cell_xy(c) for c in path2])
    ax.plot(xs1, ys1, "-o", color="tab:blue", label="Drone 1", alpha=0.8)
    ax.plot(xs2, ys2, "-o", color="tab:orange", label="Drone 2", alpha=0.8)
    ax.plot(xs1[0], ys1[0], "s", color="tab:blue", markersize=14, mfc="none", mew=2)
    ax.plot(xs2[0], ys2[0], "s", color="tab:orange", markersize=14, mfc="none", mew=2)
    ux, uy = zip(*[cell_xy(u) for u in users])
    ax.scatter(ux, uy, marker="*", s=250, color="red", label="Users", zorder=5)

    ax.set_xlim(0, GRID_SIZE)
    ax.set_ylim(0, GRID_SIZE)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title("Drone trajectories (square = start)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.05), ncol=3, fontsize=8)

    # --- Panel 2: coverage probability & sigma over the episode ----------
    ax = axes[1]
    periods = range(len(probs))
    ax.plot(periods, probs, "-o", color="tab:green", label="P_comm")
    ax.set_xlabel("period")
    ax.set_ylabel("mean coverage probability", color="tab:green")
    ax.set_ylim(0, 1.05)
    ax2 = ax.twinx()
    ax2.plot(range(len(sigmas)), sigmas, "-o", color="tab:purple", label="sigma", alpha=0.6)
    ax2.set_ylabel("sigma", color="tab:purple")
    ax.set_title("Evaluation episode: coverage & sigma")

    # --- Panel 3: training progress ---------------------------------------
    ax = axes[2]
    window = 20
    if len(training_curve) >= window:
        smoothed = np.convolve(training_curve, np.ones(window) / window, mode="valid")
        ax.plot(smoothed, color="tab:blue")
    else:
        ax.plot(training_curve, color="tab:blue")
    ax.set_xlabel("training episode")
    ax.set_ylabel("final-period P_comm (smoothed)")
    ax.set_title("Learning curve")

    fig.tight_layout()
    fig.savefig(outfile, dpi=150)
    plt.close(fig)


# --------------------------------------------------------------------------
if __name__ == "__main__":
    env, agent, training_curve = train(n_episodes=800, periods_per_episode=15)

    print(f"Trained {len(training_curve)} episodes.")
    print(f"Avg final-period P_comm, last 50 episodes: {np.mean(training_curve[-50:]):.3f}")
    print(f"Final exploration rate (epsilon): {agent.epsilon:.3f}")
    print(f"Q-table size (states visited): {len(agent.q)}")

    users = (1, 2, 5)
    path1, path2, probs, sigmas = evaluate(agent, users=list(users))

    print(f"\nGreedy evaluation, users at cells {users}, drones starting at 1 and 16:")
    for t in range(len(probs) - 1):
        print(f"  period {t+1:2d}: drone1={path1[t+1]:2d}  drone2={path2[t+1]:2d}  "
              f"P_comm={probs[t+1]:.3f}  sigma={sigmas[t+1]:.2f}")

    out_path = "drone_sim_result_.png"
    plot_simulation(path1, path2, probs, sigmas, users, training_curve, out_path)
    print(f"\nSaved plot to {out_path}")


Trained 800 episodes.
Avg final-period P_comm, last 50 episodes: 0.421
Final exploration rate (epsilon): 0.050
Q-table size (states visited): 632

Greedy evaluation, users at cells (1, 2, 5), drones starting at 1 and 16:
  period  1: drone1= 1  drone2=12  P_comm=0.867  sigma=1.50
  period  2: drone1= 1  drone2= 8  P_comm=0.867  sigma=1.50
  period  3: drone1= 1  drone2= 4  P_comm=0.867  sigma=1.50
  period  4: drone1= 2  drone2= 4  P_comm=0.814  sigma=1.50
  period  5: drone1= 2  drone2= 4  P_comm=0.814  sigma=1.50
  period  6: drone1= 2  drone2= 4  P_comm=0.814  sigma=1.50
  period  7: drone1= 2  drone2= 4  P_comm=0.814  sigma=1.50
  period  8: drone1= 2  drone2= 4  P_comm=0.814  sigma=1.50
  period  9: drone1= 2  drone2= 4  P_comm=0.814  sigma=1.50
  period 10: drone1= 2  drone2= 4  P_comm=0.814  sigma=1.70
  period 11: drone1= 2  drone2= 4  P_comm=0.849  sigma=2.35
  period 12: drone1= 2  drone2= 4  P_comm=0.916  sigma=2.64
  period 13: drone1= 2  drone2= 4  P_comm=0.932  sigma=2.27